# Assemble and Check the Firm Summaries

| Item | Description |
|---|---|
| Goal | Put the output of every T3.3.4 run into the firm-summary table and check each run. Every firm of the set has one summary, none is empty, all are within the encoder limit, and each is built from the right number of yearly summaries. |
| Input | `firm_summaries/<set>/<family>/<version>/firms.jsonl` and `firms.json` from T3.3.4 in the T3.3 Interim folder; `filings.parquet` from T3.1.1. |
| Output | `firm_summaries.parquet` in the T3.3 Processed folder: one row per firm, set, family, and prompt version. Under `R - Results/tables/...`: `table_firm_summary_checks.csv` (one row per run), `table_example_firm_summaries.csv`, and `table_firm_summaries_all_years.csv`, every firm-level summary of the main sample in full. |
| Counted, not enforced | Summaries trimmed to the limit by the last-resort rule, length by number of filings (a firm with one filing gets a rewrite of one yearly summary), and whether the summary refers to its own inputs. |
| Run | Run 2026-10-31; the table equals the one assembled in October 2026, 1,899 rows. |


## Step 1. Import packages


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.3.5 - Check - Assemble the Firm Summaries"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
FIRM_FOLDER = INTERIM_FOLDER / "firm_summaries"
FILINGS_FILE = SCRATCH_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.1 - Sample - Firms and 10-K Item 1 Filings" / "Processed" / "filings.parquet"

MAIN_SET = "all_years"
MAX_BGE_TOKENS = 512  # the encoder's input limit every summary must respect
SAMPLE_FLAG = {"three_years": "in_three_years", "all_years": "in_all_years"}
REFERS_TO_INPUT = re.compile(  # the pattern T3.3.3 counts on the filing summaries
    r"\b(the|these|provided) (provided |segment |yearly )?summar(y|ies)\b|\b(this|the) (text|segment)s? (does|do) not\b"
    r"|contained within this (text|segment)|not (described|mentioned|provided|contained) in (the|this|these) (text|segment|summar)", re.I)
RANDOM_SEED = 0
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (PROCESSED_FOLDER, TABLE_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Read every run and check it

One line per firm in `firms.jsonl`; a firm summarized in two sessions appears twice and the later record wins. Every firm with a filing in the set has exactly one summary, `n_filings` equals the firm's filings in the set, and no summary is empty or over the limit.


In [3]:
filings = pd.read_parquet(FILINGS_FILE)
tables, check_rows = [], []
for record_file in sorted(FIRM_FOLDER.glob("*/*/*/firms.json")):
    set_name, family, version = record_file.parts[-4], record_file.parts[-3], record_file.parts[-2]
    summaries = pd.DataFrame([json.loads(line) for line in record_file.with_name("firms.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()])
    record = json.loads(record_file.read_text(encoding="utf-8"))
    n_duplicates = int(summaries.duplicated("gvkey", keep="last").sum())
    summaries = summaries.drop_duplicates("gvkey", keep="last").reset_index(drop=True)
    in_set = filings.loc[filings[SAMPLE_FLAG[set_name]].eq(1)]
    expected = in_set.groupby("gvkey").agg(n_filings_expected=("accessionNo", "size"), years_expected=("fyear", lambda s: sorted(int(y) for y in s)))
    assert set(summaries["gvkey"]) == set(expected.index) and summaries["gvkey"].is_unique, (set_name, version)
    summaries = summaries.merge(expected, left_on="gvkey", right_index=True, how="left")
    assert summaries["n_filings"].eq(summaries["n_filings_expected"]).all() and all(list(a) == b for a, b in zip(summaries["years"], summaries["years_expected"]))
    assert summaries["summary"].str.strip().ne("").all() and summaries["n_bge_tokens"].le(MAX_BGE_TOKENS).all() and summaries["within_bge_limit"].all()
    assert summaries["prompt_version"].eq(version).all() and summaries["family"].eq(family).all() and summaries["set"].eq(set_name).all()
    summaries["trimmed"] = summaries["trimmed"].fillna(False).astype(bool) if "trimmed" in summaries else False
    summaries["refers_to_input"] = summaries["summary"].map(lambda text: REFERS_TO_INPUT.search(text) is not None)
    tables.append(summaries.drop(columns=["n_filings_expected", "years_expected"]))
    check_rows.append({"set": set_name, "family": family, "prompt_version": version, "n_firms": len(summaries), "n_filings_total": int(summaries["n_filings"].sum()),
                       "duplicates_dropped": n_duplicates, "words_median": float(summaries["n_words"].median()), "words_min": int(summaries["n_words"].min()), "words_max": int(summaries["n_words"].max()),
                       "bge_tokens_max": int(summaries["n_bge_tokens"].max()), "re_asked": int(summaries["reduce_rounds"].gt(1).sum()), "trimmed": int(summaries["trimmed"].sum()),
                       "refers_to_input_share": float(summaries["refers_to_input"].mean()), "gpu": record["gpu"], "last_session_finished": record["finished"]})
    print(f"{set_name:12s} {version}: {len(summaries):,} firms, {int(summaries['n_filings'].sum()):,} yearly summaries; words min {summaries['n_words'].min()}, median {int(summaries['n_words'].median())}, max {summaries['n_words'].max()}; "
          f"encoder tokens max {summaries['n_bge_tokens'].max()}; re-asked {int(summaries['reduce_rounds'].gt(1).sum())}, trimmed {int(summaries['trimmed'].sum())}, refers to inputs {summaries['refers_to_input'].mean():.1%}")
checks = pd.DataFrame(check_rows)
checks.to_csv(TABLE_FOLDER / "table_firm_summary_checks.csv", index=False)
display(checks)


all_years    item1-mapreduce-1.3: 1,058 firms, 18,779 yearly summaries; words min 126, median 248, max 401; encoder tokens max 511; re-asked 2, trimmed 0, refers to inputs 1.4%
three_years  item1-mapreduce-1.3: 841 firms, 2,072 yearly summaries; words min 128, median 258, max 392; encoder tokens max 512; re-asked 13, trimmed 0, refers to inputs 3.4%


,set,family,prompt_version,n_firms,n_filings_total,duplicates_dropped,words_median,words_min,words_max,bge_tokens_max,re_asked,trimmed,refers_to_input_share,gpu,last_session_finished
0,all_years,summary_lines_of_business,item1-mapreduce-1.3,1058,18779,0,248.0,126,401,511,2,0,0.014178,NVIDIA A40,2026-10-26T22:46:06+00:00
1,three_years,summary_lines_of_business,item1-mapreduce-1.3,841,2072,0,258.0,128,392,512,13,0,0.034483,NVIDIA A40,2026-10-26T02:01:07+00:00


## Step 4. Build the table

One row per firm, set, family, and prompt version.


In [4]:
table_columns = ["gvkey", "conm", "set", "family", "prompt_version", "n_filings", "years", "summary", "n_words", "n_bge_tokens", "reduce_rounds", "trimmed", "refers_to_input", "run_started"]
table = pd.concat(tables, ignore_index=True)[table_columns]
table["years"] = table["years"].map(lambda years: [int(y) for y in years])
assert not table.duplicated(["gvkey", "set", "family", "prompt_version"]).any()
main = table.loc[table["set"].eq(MAIN_SET)].reset_index(drop=True)
print(f"Table: {len(table):,} rows; main sample {MAIN_SET}: {len(main):,} firms")
print(table.groupby(["set", "family", "prompt_version"]).size().rename("rows").to_string())


Table: 1,899 rows; main sample all_years: 1,058 firms
set          family                     prompt_version     
all_years    summary_lines_of_business  item1-mapreduce-1.3    1058
three_years  summary_lines_of_business  item1-mapreduce-1.3     841


## Step 5. Length by number of filings

A firm with one filing gets a rewrite of a single yearly summary, so its length against the yearly summary shows whether the rewrite padded.


In [5]:
by_filings = main.groupby("n_filings").agg(firms=("gvkey", "size"), words_median=("n_words", "median"), words_max=("n_words", "max"), bge_median=("n_bge_tokens", "median"), refers_to_input=("refers_to_input", "sum"))
display(by_filings.T)
print(f"Main sample: summaries referring to their own inputs: {int(main['refers_to_input'].sum()):,} of {len(main):,} ({main['refers_to_input'].mean():.1%}); firms with all 27 fiscal years: {int(main['n_filings'].eq(27).sum()):,}")


n_filings,1,2,3,4,5,6,7,8,9,10,...,18,19,20,21,22,23,24,25,26,27
firms,32.0,38.0,19.0,18.0,23.0,35.0,35.0,27.0,37.0,28.0,...,25.0,28.0,22.0,27.0,23.0,20.0,23.0,27.0,46.0,340.0
words_median,208.0,262.5,262.0,263.5,252.0,258.0,249.0,264.0,255.0,260.0,...,246.0,246.5,247.0,245.0,250.0,245.5,244.0,247.0,246.0,245.0
words_max,289.0,385.0,353.0,340.0,401.0,376.0,348.0,336.0,356.0,328.0,...,371.0,342.0,333.0,310.0,336.0,298.0,285.0,350.0,297.0,364.0
bge_median,276.5,348.5,342.0,358.0,343.0,336.0,332.0,345.0,326.0,351.5,...,334.0,326.0,325.5,321.0,333.0,317.5,312.0,331.0,322.5,325.0
refers_to_input,4.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0


Main sample: summaries referring to their own inputs: 15 of 1,058 (1.4%); firms with all 27 fiscal years: 340


## Step 6. Show examples from the main sample

One firm with one, two, and three filings, plus the shortest and the longest summary.


In [6]:
rng = np.random.default_rng(RANDOM_SEED)
picks = [main.loc[main["n_filings"].eq(n)].iloc[[rng.integers(int(main["n_filings"].eq(n).sum()))]] for n in (1, 2, 3)]
examples = pd.concat(picks + [main.nsmallest(1, "n_words"), main.nlargest(1, "n_words")]).drop_duplicates("gvkey")
examples = examples[["gvkey", "conm", "n_filings", "years", "n_words", "n_bge_tokens", "reduce_rounds", "refers_to_input", "summary"]]
examples.to_csv(TABLE_FOLDER / "table_example_firm_summaries.csv", index=False)
for example in examples.itertuples(index=False):
    print(f"=== {example.conm} | {example.n_filings} filings {list(example.years)} | {example.n_words} words ===\n{example.summary}\n")


=== UNION PACIFIC RESOURCES GRP | 1 filings [1999] | 215 words ===
Union Pacific Resources Group Inc. is primarily engaged in the exploration, development, and production of natural gas, natural gas liquids, and crude oil across the United States, Canada, Guatemala, and Venezuela, while also operating in the hard minerals sector through non-operated joint ventures and royalty arrangements for coal, industrial minerals, and trona mines in Wyoming. The company's core operating areas include U.S. Onshore, U.S. Offshore, Canada, and Other International. It sells its natural gas and crude oil production to customers, with a significant portion of domestic sales directed to Duke Energy Field Services, Inc. under a long-term agreement, while generating additional revenue from mineral rights and royalty arrangements associated with its coal and industrial mineral operations. The Minerals segment exploits fee mineral interests in coal, trona, and industrial minerals, producing coal, soda ash de

## Step 7. Save the table and the readable copy of the main sample

The CSV holds every firm-level summary of the all-years sample in full, for reading without code.


In [7]:
table.to_parquet(PROCESSED_FOLDER / "firm_summaries.parquet", index=False)
readable = main[["gvkey", "conm", "n_filings", "years", "n_words", "summary"]].copy()
readable["years"] = readable["years"].map(lambda years: ", ".join(str(y) for y in years))
readable.to_csv(TABLE_FOLDER / "table_firm_summaries_all_years.csv", index=False)
print(f"Saved firm_summaries.parquet ({len(table):,} rows), table_firm_summaries_all_years.csv ({len(readable):,} rows), table_firm_summary_checks.csv, and table_example_firm_summaries.csv")


Saved firm_summaries.parquet (1,899 rows), table_firm_summaries_all_years.csv (1,058 rows), table_firm_summary_checks.csv, and table_example_firm_summaries.csv
